# Tutorial - SageMath - CoCalc - Interactive Simplex Method - Simplex Duas Fases

## 1. Configuração do Ambiente

Nesta etapa, configuramos o ambiente do CoCalc para voltar a renderizar o LaTeX.

Essa célula deve ser executada apenas:
- Ao abrir o notebook pela primeira vez
- Ou quando o ambiente for reiniciado/desconectado

In [1]:
from sage.numerical.interactive_simplex_method import (
    InteractiveLPProblem, 
    InteractiveLPProblemStandardForm,
    LPDictionary,
)
from IPython.display import Latex, display
from sage.misc.html import HtmlFragment
import re

def _repr_latex_(self):
    tex = self._latex_()

    tex = tex.replace(r"\displaystyle", "")
    tex = tex.replace(r"\mspace{-6mu}", "")
    tex = tex.replace(r"\end{aligned}", r"\end{array}")
    tex = tex.replace(
        r"\end{aligned} \\",
        r"\end{array} \\"
    )

    return r"\[" + tex + r"\]"

InteractiveLPProblem._repr_latex_ = _repr_latex_

if not hasattr(InteractiveLPProblemStandardForm, "_sage_original_run_simplex_method"):
    InteractiveLPProblemStandardForm._sage_original_run_simplex_method = (
        InteractiveLPProblemStandardForm.run_simplex_method
    )


def show_dictionary_latex(d):

    tex = d._latex_()

    tex = tex.replace(
        r"\renewcommand{\arraystretch}{1.5} %notruncate",
        ""
    )
    tex = tex.replace(r"\mspace{-6mu}", "")

    return r"\[" + tex + r"\]"


def run_simplex_method_problem_latex(self):

    output = []

    d = self.initial_dictionary()

    if not d.is_feasible():

        # Substitui d._html_()
        tex = show_dictionary_latex(d)
        display(Latex(tex))

        output.append(
            "The initial dictionary is infeasible, solving auxiliary problem."
        )

        ad = self.auxiliary_problem().initial_dictionary()

        ad.enter(self.auxiliary_variable())

        ad.leave(
            min(
                zip(
                    ad.constant_terms(),
                    ad.basic_variables()
                )
            )[1]
        )

        R = ad.run_simplex_method()

        output.append(R)

        if ad.objective_value() < 0:

            output.append("The original problem is infeasible.")

            self._final_dictionary = ad

        else:

            output.append("Back to the original problem.")

            d = self.feasible_dictionary(ad)


    if d.is_feasible():

        R = d.run_simplex_method()

        output.append(R)

        if d.is_optimal():

            v = d.objective_value()

            if self._is_negative:
                v = -v

            output.append(
                ("The optimal value: ${}$. "
                 "An optimal solution: ${}$.")
                .format(
                    latex(v),
                    latex(d.basic_solution())
                )
            )

        self._final_dictionary = d

    return HtmlFragment("\n".join(map(str, output)))


InteractiveLPProblemStandardForm.run_simplex_method = (
    run_simplex_method_problem_latex
)

def _repr_latex_dictionary_(self):

    tex = self._latex_()

    tex = tex.replace(
        r"\renewcommand{\arraystretch}{1.5} %notruncate",
        ""
    )

    tex = tex.replace(
        r"\mspace{-6mu}",
        ""
    )

    return r"\[" + tex + r"\]"


LPDictionary._repr_latex_ = _repr_latex_dictionary_

if not hasattr(LPDictionary, "_sage_original_run_simplex_method"):
    LPDictionary._sage_original_run_simplex_method = (
        LPDictionary.run_simplex_method
    )


def run_simplex_method_latex(self, *args, **kwargs):

    # Chama SEMPRE o método original do Sage,
    # e não o método que foi instalado pelo patch.
    R = self._sage_original_run_simplex_method(
        *args,
        **kwargs
    )

    tex = str(R)

    # Remove equation*
    tex = tex.replace(
        r"\begin{equation*}",
        ""
    )

    tex = tex.replace(
        r"\end{equation*}",
        ""
    )

    # Remove arraystretch
    tex = tex.replace(
        r"\renewcommand{\arraystretch}{1.5} %notruncate",
        ""
    )

    # Remove mspace
    tex = tex.replace(
        r"\mspace{-6mu}",
        ""
    )

    return HtmlFragment(tex)


LPDictionary.run_simplex_method = run_simplex_method_latex

class simplex_duas_fases:
    """
    Método Simplex das Duas Fases para SageMath 10.9.

    Fluxo:

        P
        |
        v
    construir_fase_1()
        |
        v
    construir_forma_padrao_fase_1()
        |
        v
    Fase I
        |
        +---- objetivo != 0 ----> inviável
        |
        v
    expulsar_artificiais_basicas()
        |
        v
    dicionario_fase_2()
        |
        v
    Fase II
        |
        v
    solução ótima
    """

    def __init__(self, P):

        self.P = P

        self.P2 = None
        self.P3 = None

        self.D3 = None
        self.D4 = None

        self.artificiais = ()

        self.valor_fase_I = None
        self.viavel = None

        self.fase = None
        self.resultado = None

    # =========================================================
    # CONSTRUIR FASE I
    # =========================================================

    def construir_fase_1(self):

        P = self.P

        A = P.A()
        b = P.b()
        c = P.c()

        constraint_types = list(
            P.constraint_types()
        )

        variable_types = list(
            P.variable_types()
        )

        variaveis = list(
            P.decision_variables()
        )

        m = P.n_constraints()
        n = P.n_variables()

        nomes = [
            str(v)
            for v in variaveis
        ]

        if len(nomes) != n:

            nomes = [
                f"x_{i+1}"
                for i in range(n)
            ]

        A2 = [
            list(linha)
            for linha in A
        ]

        nomes2 = list(nomes)

        artificiais = []

        contador_R = 1
        contador_artificial = 1

        # =====================================================
        # Processa restrições
        # =====================================================

        for i in range(m):

            tipo = str(
                constraint_types[i]
            )

            # -------------------------------------------------
            # <=
            # -------------------------------------------------

            if tipo == "<=":

                nome_R = (
                    f"R_{contador_R}"
                )

                contador_R += 1

                nomes2.append(
                    nome_R
                )

                for linha in A2:
                    linha.append(0)

                A2[i][-1] = 1

            # -------------------------------------------------
            # >=
            # -------------------------------------------------

            elif tipo == ">=":

                nome_R = (
                    f"R_{contador_R}"
                )

                contador_R += 1

                nomes2.append(
                    nome_R
                )

                for linha in A2:
                    linha.append(0)

                A2[i][-1] = -1

                nome_artificial = (
                    f"a_{contador_artificial}"
                )

                contador_artificial += 1

                nomes2.append(
                    nome_artificial
                )

                artificiais.append(
                    nome_artificial
                )

                for linha in A2:
                    linha.append(0)

                A2[i][-1] = 1

            # -------------------------------------------------
            # =
            # -------------------------------------------------

            elif tipo in ("==", "="):

                nome_artificial = (
                    f"a_{contador_artificial}"
                )

                contador_artificial += 1

                nomes2.append(
                    nome_artificial
                )

                artificiais.append(
                    nome_artificial
                )

                for linha in A2:
                    linha.append(0)

                A2[i][-1] = 1

            else:

                raise ValueError(
                    f"Tipo de restrição inválido na "
                    f"restrição {i+1}: {tipo}"
                )

        # =====================================================
        # Objetivo da Fase I
        # =====================================================

        c2 = []

        for nome in nomes2:

            if nome in artificiais:
                c2.append(1)
            else:
                c2.append(0)

        # =====================================================
        # Todas as restrições são igualdades
        # =====================================================

        constraint_types2 = [
            "=="
            for _ in range(m)
        ]

        # =====================================================
        # Tipos das variáveis
        # =====================================================

        variable_types2 = list(
            variable_types
        )

        while len(variable_types2) < len(nomes2):

            variable_types2.append(
                ">="
            )

        # =====================================================
        # Cria P2
        # =====================================================

        self.P2 = InteractiveLPProblem(
            tuple(
                tuple(linha)
                for linha in A2
            ),
            tuple(b),
            tuple(c2),
            nomes2,
            problem_type="min",
            constraint_type=constraint_types2,
            variable_type=variable_types2
        )

        self.artificiais = tuple(
            artificiais
        )

        return (
            self.P2,
            self.artificiais
        )

    # =========================================================
    # CONSTRUIR FORMA PADRÃO DA FASE I
    # =========================================================

    def construir_forma_padrao_fase_1(self):

        P2 = self.P2
        artificiais = self.artificiais

        A2 = [
            list(linha)
            for linha in P2.A()
        ]

        b2 = tuple(
            P2.b()
        )

        nomes = [
            str(v)
            for v in P2.decision_variables()
        ]

        m = P2.n_constraints()

        variaveis_basicas = []
        indices_basicos = []

        # =====================================================
        # Identificar base inicial
        # =====================================================

        for j, nome in enumerate(nomes):

            coluna = [
                A2[i][j]
                for i in range(m)
            ]

            for linha_pivo in range(m):

                esperado = [
                    1 if i == linha_pivo else 0
                    for i in range(m)
                ]

                if coluna == esperado:

                    if j not in indices_basicos:

                        indices_basicos.append(
                            j
                        )

                        variaveis_basicas.append(
                            nome
                        )

                    break

        if len(indices_basicos) != m:

            raise ValueError(
                "Não foi possível identificar uma base "
                "completa para a Fase I."
            )

        # =====================================================
        # Não-básicas
        # =====================================================

        indices_nao_basicos = [
            j
            for j in range(len(nomes))
            if j not in indices_basicos
        ]

        variaveis_nao_basicas = [
            nomes[j]
            for j in indices_nao_basicos
        ]

        # =====================================================
        # Matriz
        # =====================================================

        A3 = []

        for i in range(m):

            A3.append([
                A2[i][j]
                for j in indices_nao_basicos
            ])

        # =====================================================
        # Objetivo da Fase I
        # =====================================================

        c3 = [
            0
            for _ in variaveis_nao_basicas
        ]

        objective_constant_term = 0

        indices_artificiais = [
            j
            for j in indices_basicos
            if nomes[j] in artificiais
        ]

        for indice_basico in indices_artificiais:

            coluna = [
                A2[i][indice_basico]
                for i in range(m)
            ]

            linha_base = coluna.index(1)

            objective_constant_term -= (
                b2[linha_base]
            )

            for j, indice_nb in enumerate(
                indices_nao_basicos
            ):

                c3[j] += (
                    A2[linha_base][indice_nb]
                )

        # =====================================================
        # P3
        # =====================================================

        self.P3 = InteractiveLPProblemStandardForm(
            tuple(
                tuple(linha)
                for linha in A3
            ),
            tuple(b2),
            tuple(c3),
            variaveis_nao_basicas,
            slack_variables=variaveis_basicas,
            objective_constant_term=(
                objective_constant_term
            )
        )

        return self.P3

    # =========================================================
    # IDENTIFICAR ARTIFICIAIS
    # =========================================================

    def identificar_artificiais(self, N=None):

        if N is None:

            return tuple(
                self.artificiais
            )

        return tuple(
            x
            for x in N
            if str(x).startswith("a_")
        )

    # =========================================================
    # ARTIFICIAIS BÁSICAS
    # =========================================================

    def artificiais_basicas(self, D=None, artificiais=None):

        if D is None:
            D = self.D3

        if artificiais is None:
            artificiais = self.artificiais

        artificiais = tuple(
            str(a)
            for a in artificiais
        )

        B = tuple(
            D.basic_variables()
        )

        return tuple(
            a
            for a in B
            if str(a) in artificiais
        )

    # =========================================================
    # EXPULSAR ARTIFICIAIS BÁSICAS
    # =========================================================

    def expulsar_artificiais_basicas(self,D=None,artificiais=None):
        """
        Expulsa da base as variáveis artificiais.

        Uma variável artificial NUNCA pode ser escolhida
        como variável entrante.

        O dicionário original D não é alterado.
        """

        if D is None:
            D = self.D3

        if artificiais is None:
            artificiais = self.artificiais

        artificiais = tuple(
            str(a)
            for a in artificiais
        )

        # =====================================================
        # Copiar D
        # =====================================================

        A, b, c, v, B, N, z = (
            D._AbcvBNz
        )

        D_atual = LPDictionary(
            A,
            b,
            c,
            v,
            tuple(B),
            tuple(N),
            z
        )

        # =====================================================
        # Expulsar artificiais
        # =====================================================

        while True:

            B_atual = tuple(
                D_atual.basic_variables()
            )

            N_atual = tuple(
                D_atual.nonbasic_variables()
            )

            # =================================================
            # Usar método da própria classe
            # =================================================

            artificiais_na_base = (
                self.artificiais_basicas(
                    D_atual,
                    artificiais
                )
            )

            # -------------------------------------------------
            # Nenhuma artificial na base
            # -------------------------------------------------

            if not artificiais_na_base:
                break

            # -------------------------------------------------
            # Escolher artificial
            # -------------------------------------------------

            a = artificiais_na_base[0]

            i = B_atual.index(a)

            A_atual = (
                D_atual._AbcvBNz[0]
            )

            linha = A_atual.row(i)

            # -------------------------------------------------
            # Procurar variável NÃO-ARTIFICIAL para entrar
            # -------------------------------------------------

            candidatos = [
                x
                for j, x in enumerate(N_atual)
                if str(x) not in artificiais
                and linha[j] != 0
            ]

            # -------------------------------------------------
            # Não existe variável para expulsar artificial
            # -------------------------------------------------

            if not candidatos:

                raise ValueError(
                    f"A variável artificial {a} permanece básica "
                    "com valor zero, mas não existe variável "
                    "não-artificial capaz de entrar na base. "
                    "A restrição correspondente é redundante."
                )

            # -------------------------------------------------
            # Escolher variável entrante
            # -------------------------------------------------

            x_entrante = candidatos[0]

            print(
                f"\nExpulsando {a}: "
                f"entra {x_entrante}, "
                f"sai {a}"
            )

            # -------------------------------------------------
            # Pivot
            # -------------------------------------------------

            D_atual.enter(
                x_entrante
            )

            D_atual.leave(
                a
            )

            D_atual.update()

            # -------------------------------------------------
            # Mostrar dicionário produzido
            # -------------------------------------------------

            show(
                D_atual
            )

        return D_atual

    # =========================================================
    # DICIONÁRIO FASE II
    # =========================================================

    def dicionario_fase_2(self,P,D3,artificiais=None):
        """
        Constrói o dicionário inicial da Fase II a partir do
        dicionário final da Fase I.

        As variáveis artificiais:

            1. são expulsas da base;
            2. nunca podem entrar na base durante essa operação;
            3. são removidas das não-básicas;
            4. não podem aparecer em D4.
        """

        # =====================================================
        # 1. Identificar artificiais
        # =====================================================

        if artificiais is None:

            A0, b0, c10, v10, B0, N0, z0 = (
                D3._AbcvBNz
            )

            B0 = tuple(B0)
            N0 = tuple(N0)

            artificiais = (
                self.identificar_artificiais(
                    B0 + N0
                )
            )

        else:

            artificiais = tuple(
                artificiais
            )

        artificiais = tuple(
            str(a)
            for a in artificiais
        )

        if not artificiais:

            raise ValueError(
                "Nenhuma variável artificial foi encontrada."
            )

        # =====================================================
        # 2. Expulsar artificiais básicas
        #
        # Usa o método já existente na classe.
        # =====================================================

        D_atual = (
            self.expulsar_artificiais_basicas(
                D3,
                artificiais
            )
        )

        # =====================================================
        # 3. Extrair dicionário após expulsão
        # =====================================================

        A, b, c1, v1, B, N, z1 = (
            D_atual._AbcvBNz
        )

        B = tuple(B)
        N = tuple(N)

        # =====================================================
        # 4. Verificação:
        #    nenhuma artificial pode estar na base
        # =====================================================

        artificiais_na_base = tuple(
            x
            for x in B
            if str(x) in artificiais
        )

        if artificiais_na_base:

            raise RuntimeError(
                "Ainda existem variáveis artificiais "
                "na base após a expulsão: "
                + str(artificiais_na_base)
            )

        # =====================================================
        # 5. Remover artificiais das não-básicas
        # =====================================================

        N2 = tuple(
            x
            for x in N
            if str(x) not in artificiais
        )

        # =====================================================
        # 6. Colunas que permanecem
        # =====================================================

        colunas_manter = [
            j
            for j, x in enumerate(N)
            if str(x) not in artificiais
        ]

        A2 = A.matrix_from_columns(
            colunas_manter
        )

        B2 = B

        b2 = vector(
            D_atual.base_ring(),
            b
        )

        # =====================================================
        # 7. Verificação final antes de construir D4
        # =====================================================

        artificiais_restantes = tuple(
            x
            for x in B2 + N2
            if str(x) in artificiais
        )

        if artificiais_restantes:

            raise RuntimeError(
                "ERRO: variáveis artificiais ainda "
                "estão presentes antes da construção "
                "de D4: "
                + str(artificiais_restantes)
            )

        # =====================================================
        # 8. Problema original
        # =====================================================

        A_original, b_original, c_original, x_original = (
            P.Abcx()
        )

        # =====================================================
        # 9. Reconstruir objetivo original
        # =====================================================

        c2 = vector(
            D_atual.base_ring(),
            [0] * len(N2)
        )

        v2 = D_atual.base_ring()(
            P._constant_term
        )

        for cj, xj in zip(
            c_original,
            x_original
        ):

            # -------------------------------------------------
            # Variável original não-básica
            # -------------------------------------------------

            if xj in N2:

                j = N2.index(xj)

                c2[j] += cj

            # -------------------------------------------------
            # Variável original básica
            # -------------------------------------------------

            elif xj in B2:

                i = B2.index(xj)

                v2 += (
                    cj * b2[i]
                )

                c2 -= (
                    cj * A2.row(i)
                )

            # -------------------------------------------------
            # Variável não encontrada
            # -------------------------------------------------

            else:

                raise ValueError(
                    f"A variável original {xj} não pertence "
                    "ao dicionário da Fase I após a eliminação "
                    "das artificiais."
                )

        # =====================================================
        # 10. MIN -> MAX
        # =====================================================

        if P.problem_type() == "min":

            c2 = -c2
            v2 = -v2

        # =====================================================
        # 11. Criar D4
        # =====================================================

        D4 = LPDictionary(
            A2,
            b2,
            c2,
            v2,
            B2,
            N2,
            z1
        )

        # =====================================================
        # 12. Verificação definitiva
        #
        # D4 NÃO PODE conter artificiais.
        # =====================================================

        B4 = tuple(
            D4.basic_variables()
        )

        N4 = tuple(
            D4.nonbasic_variables()
        )

        artificiais_restantes = tuple(
            x
            for x in B4 + N4
            if str(x) in artificiais
        )

        if artificiais_restantes:

            raise RuntimeError(
                "ERRO: artificiais ainda presentes em D4: "
                + str(artificiais_restantes)
            )

#        print(
#            "\nDicionário inicial da Fase II:"
#        )

        # show(D4)

        return D4

    # =========================================================
    # SIMPLEX - MAIOR COEFICIENTE POSITIVO
    # =========================================================

    def simplex_maior_coeficiente(self,P,max_iter=100):
        """
        Executa o método Simplex escolhendo como variável
        entrante aquela que possui o maior coeficiente positivo
        na função objetivo.

        Critério de entrada:

            maior coeficiente positivo.

        Critério de saída:

            teste da razão mínima, através de
            possible_leaving().

        Em caso de empate na saída, escolhe a menor variável.

        P pode ser:

            - um InteractiveLPProblemStandardForm;
            - um LPDictionary.
        """

        # =====================================================
        # Obter dicionário inicial
        # =====================================================

        if isinstance(P, LPDictionary):

            D = P

        else:

            D = P.initial_dictionary()

        print()
        print("=" * 60)
        print("DICIONÁRIO INICIAL")
        print("=" * 60)

        display(D)

        # =====================================================
        # Iterações
        # =====================================================

        for k in range(1, max_iter + 1):

            # -------------------------------------------------
            # Verificar ótimo
            # -------------------------------------------------

            if D.is_optimal():

#                print()
#                print(
#                    "Solução ótima encontrada "
#                    f"na iteração {k-1}."
#                )

                break

            # -------------------------------------------------
            # Variáveis que podem entrar
            # -------------------------------------------------

            candidatas = D.possible_entering()

            if not candidatas:

                raise ValueError(
                    "Não existem variáveis candidatas "
                    "a entrar."
                )

            # -------------------------------------------------
            # Variáveis não-básicas
            # -------------------------------------------------

            variaveis = tuple(
                D.nonbasic_variables()
            )

            coeficientes = tuple(
                D.objective_coefficients()
            )

            # -------------------------------------------------
            # Escolher variável com MAIOR coeficiente positivo
            # -------------------------------------------------

            entrada = max(
                candidatas,
                key=lambda v:
                    coeficientes[
                        variaveis.index(v)
                    ]
            )

            coef_entrada = (
                coeficientes[
                    variaveis.index(entrada)
                ]
            )

            # -------------------------------------------------
            # Mostrar informações
            # -------------------------------------------------

            print()
            print("=" * 60)
            print(f"ITERAÇÃO {k}")
            print("=" * 60)

            print(
                "Variável que entra:",
                entrada
            )

            # -------------------------------------------------
            # Definir variável entrante
            # -------------------------------------------------

            D.enter(
                entrada
            )

            # -------------------------------------------------
            # Variáveis que podem sair
            # -------------------------------------------------

            saidas = D.possible_leaving()

            if not saidas:

                raise ValueError(
                    f"O problema é ilimitado na direção "
                    f"da variável {entrada}."
                )

            # -------------------------------------------------
            # Escolher variável que sai
            # -------------------------------------------------

            saida = min(
                saidas
            )

            print(
                "Variável que sai:",
                saida
            )

            # -------------------------------------------------
            # Pivotamento
            # -------------------------------------------------

            D.leave(
                saida
            )

            D.update()

            # -------------------------------------------------
            # Mostrar novo dicionário
            # -------------------------------------------------

            print(
                "\nNovo dicionário:"
            )

            display(D)

            print(
                "Valor da função objetivo:",
                D.objective_value()
            )

        else:

            raise RuntimeError(
                "Número máximo de iterações "
                f"({max_iter}) atingido."
            )

        return D

    # =========================================================
    # VERIFICAR FASE I
    # =========================================================

    def verifica_fase_I(self):

        self.fase = 1

        print("=" * 60)
        print("FASE I")
        print("=" * 60)

        # -----------------------------------------------------
        # Resolver Fase I
        # -----------------------------------------------------

        self.D3 = (
            self.simplex_maior_coeficiente(
                self.P3
            )
        )

        # -----------------------------------------------------
        # Mostrar dicionário final
        # -----------------------------------------------------

        print(
            "\nDicionário final da Fase I:"
        )

        show(
            self.D3
        )

        self.valor_fase_I = (
            self.D3.objective_value()
        )

        print(
            "\nValor da função objetivo da Fase I:"
        )

        show(
            self.valor_fase_I
        )

        if self.valor_fase_I != 0:

            self.viavel = False

            print(
                "\nProblema original inviável."
            )

            return False

        self.viavel = True

        print(
            "\nProblema original viável. "
            "Siga para a Fase II."
        )

        return True

    # =========================================================
    # RESOLVER
    # =========================================================

    def resolver(self):

        # =====================================================
        # FASE I
        # =====================================================

        self.fase = 1

        print("=" * 60)
        print("FASE I")
        print("=" * 60)

        # -----------------------------------------------------
        # Construir Fase I
        # -----------------------------------------------------

        self.construir_fase_1()

        print(
            "\nProblema da Fase I (P2):"
        )

        display(
            self.P2
        )

        # -----------------------------------------------------
        # Construir forma padrão da Fase I
        # -----------------------------------------------------

        self.construir_forma_padrao_fase_1()

        # -----------------------------------------------------
        # Resolver Fase I
        # -----------------------------------------------------

        self.D3 = (
            self.simplex_maior_coeficiente(
                self.P3
            )
        )

        # -----------------------------------------------------
        # Mostrar dicionário final da Fase I
        # -----------------------------------------------------

#        print(
#            "\nDicionário final da Fase I:"
#        )

#        display(self.D3)

        # -----------------------------------------------------
        # Valor da função objetivo da Fase I
        # -----------------------------------------------------

        self.valor_fase_I = (
            self.D3.objective_value()
        )
#
#        print(
#            "\nValor da função objetivo da Fase I:"
#        )

#        show(
#            self.valor_fase_I
#        )

        # =====================================================
        # VERIFICAR VIABILIDADE
        # =====================================================

        if self.valor_fase_I != 0:

            self.viavel = False
            self.resultado = None

            print(
                "\nProblema original inviável."
            )

            return None

        # =====================================================
        # PROBLEMA VIÁVEL
        # =====================================================

        self.viavel = True

        print(
            "\nProblema original viável. "
            "Siga para a Fase II."
        )

        # =====================================================
        # FASE II
        # =====================================================

        self.fase = 2

        print()
        print("=" * 60)
        print("FASE II")
        print("=" * 60)

        # -----------------------------------------------------
        # Construir D4
        # -----------------------------------------------------

        self.D4 = (
            self.dicionario_fase_2(
                self.P,
                self.D3,
                self.artificiais
            )
        )

        # -----------------------------------------------------
        # Resolver Fase II
        # -----------------------------------------------------

        self.D4 = (
            self.simplex_maior_coeficiente(
                self.D4
            )
        )

        # -----------------------------------------------------
        # Resultado
        # -----------------------------------------------------

        self.resultado = self.D4

        return self.D4


## 2. Exemplo de um problema viável

In [2]:
A = ([3,1],[4,3],[1,2])
b = (3,6,4)
c = (4,1)

In [3]:
P = InteractiveLPProblem(A, b, c, ["x_1", "x_2"], problem_type= "min", constraint_type= ["==", ">=","<="], variable_type= [">=", ">="])
P

LP problem (use 'view(...)' or '%display typeset' for details)

In [4]:
S = simplex_duas_fases(P)
S1 = S.resolver()

FASE I

Problema da Fase I (P2):


LP problem (use 'view(...)' or '%display typeset' for details)


DICIONÁRIO INICIAL


LP problem dictionary (use 'view(...)' or '%display typeset' for details)


ITERAÇÃO 1
Variável que entra: x_1
Variável que sai: a_1

Novo dicionário:


...

Valor da função objetivo: 0

Problema original viável. Siga para a Fase II.

FASE II

DICIONÁRIO INICIAL


LP problem dictionary (use 'view(...)' or '%display typeset' for details)


ITERAÇÃO 1
Variável que entra: R_1
Variável que sai: R_2

Novo dicionário:


LP problem dictionary (use 'view(...)' or '%display typeset' for details)

Valor da função objetivo: -17/5


## 3. Exemplo de um problema inviável

In [5]:
A = ([2,4],[2,1])
b = (1,3)
c = (10,12)
P = InteractiveLPProblem(A, b, c, ["y_1", "y_2"], problem_type= "min", constraint_type= ["<=", "=="], variable_type= [">=", ">="])
P

LP problem (use 'view(...)' or '%display typeset' for details)

In [6]:
S = simplex_duas_fases(P)
S1 = S.resolver()

FASE I

Problema da Fase I (P2):


LP problem (use 'view(...)' or '%display typeset' for details)


DICIONÁRIO INICIAL


LP problem dictionary (use 'view(...)' or '%display typeset' for details)


ITERAÇÃO 1
Variável que entra: y_1
Variável que sai: R_1

Novo dicionário:


LP problem dictionary (use 'view(...)' or '%display typeset' for details)

Valor da função objetivo: -2

Problema original inviável.
